# ListenBrainz dump → daily aggregates → local Postgres
This notebook:
1. Reads a **local** `listenbrainz-*-listens-*.tar.zst` dump file
2. Streams decompress + tar extraction (no full unzip needed)
3. Parses `.listens` JSON-lines records
4. Builds daily aggregates for tracks and artists
5. Upserts into a **local Postgres** database

In [47]:
## Install dependencies (run once per environment)
!pip -q install zstandard orjson psycopg2-binary pandas tqdm


## 1) Configure paths + database connection

```bash
export PGHOST=database-1.chm317to06o1.us-east-1.rds.amazonaws.com
export PGPORT=5432
export PGDATABASE=postgres
export PGUSER=postgres
export PGPASSWORD="$(aws secretsmanager get-secret-value \
  --secret-id 'arn:aws:secretsmanager:us-east-1:549787090008:secret:rds!db-6323faa7-77d3-4952-af08-bcd6d623f642-g3XgW6' \
  --query 'SecretString' --output text \
  --region us-east-1 | python3 -c 'import json,sys; print(json.loads(sys.stdin.read())["password"])')"
export PGPASSWORD=$(aws secretsmanager get-secret-value --secret-id 'arn:aws:secretsmanager:us-east-1:549787090008:secret:rds!db-6323faa7-77d3-4952-af08-bcd6d623f642-g3XgW6' --query SecretString --output text | jq -r '.password')
```


In [48]:
import os
from pathlib import Path

# --- Set your dump path here ---
DUMP_PATH = Path("/Users/didiermunezero/Documents/NU/Junior/DE 300/de300-2026wi-munezero/Final Project/datadumps/musicbrainz/listenbrainz-listens-dump-2433-20260217-000003-incremental.tar.zst")

assert DUMP_PATH.exists(), f"Dump not found: {DUMP_PATH.resolve()}"

# --- Postgres connection via env vars (recommended) ---
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = int(os.getenv("PGPORT", "5432"))
PGDATABASE = os.getenv("PGDATABASE", "newsfeed")
PGUSER = os.getenv("PGUSER", "newsfeed")
PGPASSWORD = os.getenv("PGPASSWORD", "newsfeed")  # set via env

print("Dump:", DUMP_PATH.name)
print("Postgres:", f"{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")


Dump: listenbrainz-listens-dump-2433-20260217-000003-incremental.tar.zst
Postgres: newsfeed@localhost:5432/newsfeed


## 2) Create tables (local Postgres)
Creates the complete schema with stats tables.
Safe to re-run (`IF NOT EXISTS`) unless you changed the schema - in that case, run the drop cell above first.


**WARNING:** Run this cell to drop existing tables and start fresh (you'll lose existing data)

In [49]:
# import psycopg2

# # Drop tables in correct order (children before parents due to foreign keys)
# drop_ddl = '''
# DROP TABLE IF EXISTS track_daily_stats CASCADE;
# DROP TABLE IF EXISTS artist_daily_stats CASCADE;
# DROP TABLE IF EXISTS track_daily_listens CASCADE;
# DROP TABLE IF EXISTS artist_daily_listens CASCADE;
# DROP TABLE IF EXISTS track_info CASCADE;
# DROP TABLE IF EXISTS artist_info CASCADE;
# DROP TABLE IF EXISTS ingestion_state CASCADE;
# '''

# conn = psycopg2.connect(
#     host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
# )
# conn.autocommit = True
# with conn.cursor() as cur:
#     cur.execute(drop_ddl)
# conn.close()

# print("All tables dropped. Run the next cell to recreate them.")

In [50]:
# import psycopg2

# ddl = '''
# CREATE TABLE IF NOT EXISTS ingestion_state (
#   id                INT PRIMARY KEY DEFAULT 1,
#   last_dump_id      TEXT,
#   last_dump_path    TEXT,
#   loaded_at         TIMESTAMPTZ NOT NULL DEFAULT now(),
#   CONSTRAINT singleton_row CHECK (id = 1)
# );
# INSERT INTO ingestion_state (id) VALUES (1)
# ON CONFLICT (id) DO NOTHING;

# CREATE TABLE IF NOT EXISTS artist_info (
#   artist_mbid TEXT PRIMARY KEY,
#   artist_name TEXT
# );

# CREATE TABLE IF NOT EXISTS track_info (
#   recording_id TEXT PRIMARY KEY,
#   track_name TEXT,
#   artist_mbids TEXT[],
#   release_name TEXT
# );

# CREATE TABLE IF NOT EXISTS artist_daily_listens (
#   day DATE NOT NULL,
#   artist_mbid TEXT NOT NULL,
#   listen_count BIGINT NOT NULL,
#   PRIMARY KEY (day, artist_mbid),
#   FOREIGN KEY (artist_mbid) REFERENCES artist_info(artist_mbid)
# );

# CREATE TABLE IF NOT EXISTS track_daily_listens (
#   day DATE NOT NULL,
#   recording_id TEXT NOT NULL,
#   listen_count BIGINT NOT NULL,
#   PRIMARY KEY (day, recording_id),
#   FOREIGN KEY (recording_id) REFERENCES track_info(recording_id)
# );

# CREATE TABLE IF NOT EXISTS artist_daily_stats (
#   day DATE NOT NULL,
#   artist_mbid TEXT NOT NULL,
#   growth_percentile FLOAT,
#   cumulative_listen_count BIGINT,
#   listen_count_past_7_days BIGINT,
#   listen_pctl_past_7_days FLOAT,
#   listen_count_past_30_days BIGINT,
#   listen_pctl_past_30_days FLOAT,
#   PRIMARY KEY (day, artist_mbid),
#   FOREIGN KEY (artist_mbid) REFERENCES artist_info(artist_mbid)
# );

# CREATE TABLE IF NOT EXISTS track_daily_stats (
#   day DATE NOT NULL,
#   recording_id TEXT NOT NULL,
#   growth_percentile FLOAT,
#   cumulative_listen_count BIGINT,
#   listen_count_past_7_days BIGINT,
#   listen_pctl_past_7_days FLOAT,
#   listen_count_past_30_days BIGINT,
#   listen_pctl_past_30_days FLOAT,
#   PRIMARY KEY (day, recording_id),
#   FOREIGN KEY (recording_id) REFERENCES track_info(recording_id)
# );

# CREATE INDEX IF NOT EXISTS idx_track_daily_day ON track_daily_listens(day);
# CREATE INDEX IF NOT EXISTS idx_artist_daily_day ON artist_daily_listens(day);
# CREATE INDEX IF NOT EXISTS idx_track_stats_day ON track_daily_stats(day);
# CREATE INDEX IF NOT EXISTS idx_artist_stats_day ON artist_daily_stats(day);
# '''

# conn = psycopg2.connect(
#     host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
# )
# conn.autocommit = True
# with conn.cursor() as cur:
#     cur.execute(ddl)
# conn.close()

# print("Tables ready.")


## 3) Stream-parse the dump and build aggregates
Parses `.listens` files inside the tarball (one JSON object per line).
Handles common casing variations like `recording_mbid` vs `Recording_mbid`.

**Speed knob:** set env var `MAX_LINES` to cap records for a quick test.
Example: `export MAX_LINES=200000`


In [51]:
import tarfile
import zstandard as zstd
import orjson
from collections import defaultdict
from datetime import datetime, timezone
from tqdm import tqdm

def day_from_unix(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).date().isoformat()

def get_any(d, *keys):
    if not isinstance(d, dict):
        return None
    for k in keys:
        if k in d:
            return d[k]
    return None

def iter_listens_lines_from_tar_zst(path: Path):
    """Yield (member_name, line_bytes) for each non-empty line in *.listens files inside tar.zst."""
    with path.open("rb") as fh:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(fh) as reader:
            with tarfile.open(fileobj=reader, mode="r|*") as tf:
                for member in tf:
                    if not member.isfile():
                        continue
                    name = member.name
                    if "/listens/" not in name or not name.endswith(".listens"):
                        continue
                    f = tf.extractfile(member)
                    if f is None:
                        continue
                    for line in f:
                        line = line.strip()
                        if line:
                            yield name, line

track_daily = defaultdict(int)    # (day, recording_id) -> count
artist_daily = defaultdict(int)   # (day, artist_mbid) -> count
track_info = {}   # recording_id -> (track_name, artist_mbids, release_name)
artist_info = {}  # artist_mbid -> artist_name (best-effort)

MAX_LINES = int(os.getenv("MAX_LINES", "0"))  # 0 = no limit

lines = 0
bad_json = 0
missing_ts = 0
missing_recording = 0
missing_artist = 0

for _, line in tqdm(iter_listens_lines_from_tar_zst(DUMP_PATH), desc="Parsing listens"):
    lines += 1
    if MAX_LINES and lines > MAX_LINES:
        break

    try:
        rec = orjson.loads(line)
    except Exception:
        bad_json += 1
        continue

    ts = rec.get("timestamp")
    if ts is None:
        missing_ts += 1
        continue

    day = day_from_unix(int(ts))

    tm = rec.get("track_metadata") or {}
    add = tm.get("additional_info") or {}

    artist_mbids = get_any(add, "artist_mbids") or []

    track_name = tm.get("track_name")
    release_name = tm.get("release_name")
    artist_name = tm.get("artist_name")

    # Create recording_id from artist_mbids and track_name
    recording_id = f"{'_'.join(artist_mbids)}_{track_name}"

    # Track daily listens
    track_daily[(day, recording_id)] += 1
    if recording_id not in track_info:
        track_info[recording_id] = (track_name, artist_mbids, release_name)

    # Artist daily listens
    if isinstance(artist_mbids, list):
        for ambid in artist_mbids:
            if not ambid:
                continue
            artist_daily[(day, ambid)] += 1
            if ambid not in artist_info and artist_name:
                artist_info[ambid] = artist_name
    else:
        if artist_mbids is None or artist_mbids == "" or artist_mbids == []:
            missing_artist += 1

    

summary = {
    "lines_parsed": lines,
    "bad_json": bad_json,
    "missing_timestamp": missing_ts,
    "missing_recording_mbid": missing_recording,
    "missing_artist_mbid": missing_artist,
    "unique_track_day_keys": len(track_daily),
    "unique_artist_day_keys": len(artist_daily),
    "unique_tracks": len({k[1] for k in track_daily.keys()}),
    "unique_artists": len({k[1] for k in artist_daily.keys()}),
}
summary


Parsing listens: 2259229it [00:16, 137644.03it/s]


{'lines_parsed': 2259229,
 'bad_json': 0,
 'missing_timestamp': 0,
 'missing_recording_mbid': 0,
 'missing_artist_mbid': 0,
 'unique_track_day_keys': 1574193,
 'unique_artist_day_keys': 17518,
 'unique_tracks': 525685,
 'unique_artists': 16707}

## 4) Quick inspection with pandas


In [52]:
import pandas as pd

df_track_daily = pd.DataFrame(
    [{"day": day, "recording_id": rid, "listen_count": cnt}
     for (day, rid), cnt in track_daily.items()]
)

df_artist_daily = pd.DataFrame(
    [{"day": day, "artist_mbid": mbid, "listen_count": cnt}
     for (day, mbid), cnt in artist_daily.items()]
)

display(df_track_daily.head())
display(df_artist_daily.head())


,day,recording_id,listen_count
0,2026-02-15,39822e8d-f24e-4f07-b51b-28b22e59fbdb_Dying Planet,1
1,2026-02-16,e21857d5-3256-4547-afb3-4b6ded592596_d7641934-...,1
2,2026-02-15,_Laurel Wreath,1
3,2026-02-15,_Cool Kids,1
4,2026-02-16,185527bf-c293-4c24-8213-ed98fb8976be_Sister,1


,day,artist_mbid,listen_count
0,2026-02-15,39822e8d-f24e-4f07-b51b-28b22e59fbdb,1
1,2026-02-16,e21857d5-3256-4547-afb3-4b6ded592596,214
2,2026-02-16,d7641934-7dfb-49ea-b948-da0b12816b12,1
3,2026-02-16,185527bf-c293-4c24-8213-ed98fb8976be,4
4,2026-02-15,e07d9474-00ea-4460-ac27-88b46b3d976e,1


In [53]:
# Top tracks / artists in this dump
display(df_track_daily.sort_values("listen_count", ascending=False).head(15))
display(df_artist_daily.sort_values("listen_count", ascending=False).head(15))


,day,recording_id,listen_count
11260,2026-02-16,_This street food is all about the extras. #Ep...,1221
12744,2026-02-16,_Turkish Link,682
48837,2023-04-10,_Eulogy,607
48836,2023-04-09,_Eulogy,593
48851,2023-04-17,_Eulogy,568
48856,2023-04-21,_Eulogy,537
48755,2023-04-02,_Eulogy,517
48862,2023-04-27,_Something To Hide,501
7398,2026-02-16,_Ferto,485
48876,2023-05-01,_Something To Hide,468


,day,artist_mbid,listen_count
87,2026-02-16,f59c5520-5f46-4d2c-b2c4-822eabf53419,490
795,2026-02-16,1557e001-08ec-405d-a0cf-6d2625e90af5,331
1641,2026-02-16,074e3847-f67f-49f9-81f1-8c8cea147e8e,319
504,2026-02-16,89ad4ac3-39f7-470e-963a-56509c546377,296
64,2026-02-16,0383dadf-2a4e-4d10-a46a-e9e041da8eb3,265
506,2026-02-16,a74b1b7f-71a5-4011-9441-d0b5e4122711,254
172,2026-02-16,eb3c021d-e056-42d6-8fad-064205a90527,231
9348,2026-02-16,44e5c9b8-252e-40be-8034-7104f9b0ea62,231
70,2026-02-16,cc197bad-dc9c-440d-a5b5-d52ba2e14234,228
21,2026-02-16,8264722b-df00-467a-858e-5c97cda169c9,223


## 5) Bulk upsert into Postgres
Uses `execute_values` for performance.
Aggregates are upserted by adding to existing counts (so you can re-run multiple dumps).


In [54]:
import psycopg2
from psycopg2.extras import execute_values
import re

def chunked(iterable, n=50_000):
    buf = []
    for x in iterable:
        buf.append(x)
        if len(buf) >= n:
            yield buf
            buf = []
    if buf:
        yield buf

# Extract dump_id from filename (e.g., "listenbrainz-listens-dump-2428-...")
dump_id_match = re.search(r'dump-(\d+)-', DUMP_PATH.name)
dump_id = dump_id_match.group(1) if dump_id_match else None

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)
conn.autocommit = False

with conn, conn.cursor() as cur:
    # Upsert artist_info
    artist_rows = [(k, v) for k, v in artist_info.items()]
    for batch in chunked(artist_rows, n=20_000):
        execute_values(
            cur,
            '''
            INSERT INTO artist_info (artist_mbid, artist_name)
            VALUES %s
            ON CONFLICT (artist_mbid) DO UPDATE
            SET artist_name = COALESCE(EXCLUDED.artist_name, artist_info.artist_name)
            ''',
            batch
        )

    # Upsert track_info
    track_rows = [(k, v[0], v[1], v[2]) for k, v in track_info.items()]
    for batch in chunked(track_rows, n=20_000):
        execute_values(
            cur,
            '''
            INSERT INTO track_info (recording_id, track_name, artist_mbids, release_name)
            VALUES %s
            ON CONFLICT (recording_id) DO UPDATE
            SET track_name = COALESCE(EXCLUDED.track_name, track_info.track_name),
                artist_mbids = COALESCE(EXCLUDED.artist_mbids, track_info.artist_mbids),
                release_name = COALESCE(EXCLUDED.release_name, track_info.release_name)
            ''',
            batch
        )

    # Upsert track_daily_listens
    track_daily_rows = [(day, rid, cnt) for (day, rid), cnt in track_daily.items()]
    for batch in chunked(track_daily_rows, n=50_000):
        execute_values(
            cur,
            '''
            INSERT INTO track_daily_listens (day, recording_id, listen_count)
            VALUES %s
            ON CONFLICT (day, recording_id) DO UPDATE
            SET listen_count = track_daily_listens.listen_count + EXCLUDED.listen_count
            ''',
            batch
        )

    # Upsert artist_daily_listens
    artist_daily_rows = [(day, mbid, cnt) for (day, mbid), cnt in artist_daily.items()]
    for batch in chunked(artist_daily_rows, n=50_000):
        execute_values(
            cur,
            '''
            INSERT INTO artist_daily_listens (day, artist_mbid, listen_count)
            VALUES %s
            ON CONFLICT (day, artist_mbid) DO UPDATE
            SET listen_count = artist_daily_listens.listen_count + EXCLUDED.listen_count
            ''',
            batch
        )

    # Update ingestion_state
    cur.execute(
        '''
        UPDATE ingestion_state
        SET last_dump_id = %s, last_dump_path = %s, loaded_at = now()
        WHERE id = 1
        ''',
        (dump_id, str(DUMP_PATH))
    )

conn.close()
print("Upserts complete.")


Upserts complete.


## 6) Compute Daily Stats using Spark
Uses PySpark to compute rolling window aggregates, growth percentiles, and cumulative counts.
Spark is ideal for:
- Window functions (rolling 7/30 day aggregates)
- Percentile calculations across large datasets
- Complex analytical queries that would be slow in raw SQL

This reads from Postgres, computes stats in Spark, and writes back to the stats tables.

In [55]:
# Install PySpark if needed
!pip -q install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create Spark session with Postgres JDBC support
spark = SparkSession.builder \
    .appName("ListenBrainz Stats") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.1") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Postgres JDBC connection properties
jdbc_url = f"jdbc:postgresql://{PGHOST}:{PGPORT}/{PGDATABASE}"
jdbc_props = {
    "user": PGUSER,
    "password": PGPASSWORD,
    "driver": "org.postgresql.Driver"
}

print("Spark session created:", spark.version)

Spark session created: 4.1.1


In [56]:
# Read artist_daily_listens from Postgres
artist_df = spark.read.jdbc(
    url=jdbc_url,
    table="artist_daily_listens",
    properties=jdbc_props
)

print(f"Loaded {artist_df.count():,} artist daily records")

# Sort by artist and day to ensure proper window ordering
artist_df = artist_df.sort("artist_mbid", "day")

# Define window specifications using row-based windows (more reliable than range-based)
artist_window_unbounded = Window.partitionBy("artist_mbid").orderBy("day").rowsBetween(Window.unboundedPreceding, 0)
artist_window_7d = Window.partitionBy("artist_mbid").orderBy("day").rowsBetween(-6, 0)  # Last 7 days (including today)
artist_window_30d = Window.partitionBy("artist_mbid").orderBy("day").rowsBetween(-29, 0)  # Last 30 days (including today)

# Compute cumulative listen count
artist_df = artist_df.withColumn(
    "cumulative_listen_count",
    F.sum("listen_count").over(artist_window_unbounded)
)

# Compute rolling 7-day and 30-day listen counts
artist_df = artist_df.withColumn(
    "listen_count_past_7_days",
    F.sum("listen_count").over(artist_window_7d)
)
artist_df = artist_df.withColumn(
    "listen_count_past_30_days",
    F.sum("listen_count").over(artist_window_30d)
)

# Compute growth (today's count / cumulative from yesterday)
artist_window_lag = Window.partitionBy("artist_mbid").orderBy("day")
artist_df = artist_df.withColumn(
    "cumulative_yesterday",
    F.lag("cumulative_listen_count", 1).over(artist_window_lag)
)
artist_df = artist_df.withColumn(
    "growth_rate",
    F.when(F.col("cumulative_yesterday").isNotNull() & (F.col("cumulative_yesterday") > 0),
           F.col("listen_count") / F.col("cumulative_yesterday")
    ).otherwise(None)
)

# Compute percentiles for growth and rolling windows
artist_df = artist_df.withColumn(
    "growth_percentile",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("growth_rate").asc_nulls_first()))
)
artist_df = artist_df.withColumn(
    "listen_pctl_past_7_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_7_days")))
)
artist_df = artist_df.withColumn(
    "listen_pctl_past_30_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_30_days")))
)

# Select final columns for artist_daily_stats
artist_stats = artist_df.select(
    "day",
    "artist_mbid",
    "growth_percentile",
    "cumulative_listen_count",
    "listen_count_past_7_days",
    "listen_pctl_past_7_days",
    "listen_count_past_30_days",
    "listen_pctl_past_30_days"
)

# Write to Postgres (upsert by overwriting partition or full table)
artist_stats.write.jdbc(
    url=jdbc_url,
    table="artist_daily_stats",
    mode="overwrite",  # Can change to "append" if you want to handle duplicates differently
    properties=jdbc_props
)

print(f"Wrote {artist_stats.count():,} artist stats records")
artist_stats.show(10)

Loaded 66,395 artist daily records


Wrote 66,395 artist stats records
+----------+--------------------+------------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|       day|         artist_mbid| growth_percentile|cumulative_listen_count|listen_count_past_7_days|listen_pctl_past_7_days|listen_count_past_30_days|listen_pctl_past_30_days|
+----------+--------------------+------------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|2025-08-13|a6096ae7-5da6-435...|               0.0|                      1|                       1|                    0.0|                        1|                     0.0|
|2025-08-13|fac55e3c-d459-45e...|               0.0|                      1|                       1|                    0.0|                        1|                     0.0|
|2025-08-13|1c26f82c-b35d-4c8...|               1.0|                    283|     

In [57]:
# Read track_daily_listens from Postgres
track_df = spark.read.jdbc(
    url=jdbc_url,
    table="track_daily_listens",
    properties=jdbc_props
)

print(f"Loaded {track_df.count():,} track daily records")

# Sort by recording and day to ensure proper window ordering
track_df = track_df.sort("recording_id", "day")

# Define window specifications using row-based windows (more reliable than range-based)
track_window_unbounded = Window.partitionBy("recording_id").orderBy("day").rowsBetween(Window.unboundedPreceding, 0)
track_window_7d = Window.partitionBy("recording_id").orderBy("day").rowsBetween(-6, 0)  # Last 7 days (including today)
track_window_30d = Window.partitionBy("recording_id").orderBy("day").rowsBetween(-29, 0)  # Last 30 days (including today)

# Compute cumulative listen count
track_df = track_df.withColumn(
    "cumulative_listen_count",
    F.sum("listen_count").over(track_window_unbounded)
)

# Compute rolling 7-day and 30-day listen counts
track_df = track_df.withColumn(
    "listen_count_past_7_days",
    F.sum("listen_count").over(track_window_7d)
)
track_df = track_df.withColumn(
    "listen_count_past_30_days",
    F.sum("listen_count").over(track_window_30d)
)

# Compute growth (today's count / cumulative from yesterday)
track_window_lag = Window.partitionBy("recording_id").orderBy("day")
track_df = track_df.withColumn(
    "cumulative_yesterday",
    F.lag("cumulative_listen_count", 1).over(track_window_lag)
)
track_df = track_df.withColumn(
    "growth_rate",
    F.when(F.col("cumulative_yesterday").isNotNull() & (F.col("cumulative_yesterday") > 0),
           F.col("listen_count") / F.col("cumulative_yesterday")
    ).otherwise(None)
)

# Compute percentiles for growth and rolling windows
track_df = track_df.withColumn(
    "growth_percentile",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("growth_rate").asc_nulls_first()))
)
track_df = track_df.withColumn(
    "listen_pctl_past_7_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_7_days")))
)
track_df = track_df.withColumn(
    "listen_pctl_past_30_days",
    F.percent_rank().over(Window.partitionBy("day").orderBy(F.col("listen_count_past_30_days")))
)

# Select final columns for track_daily_stats
track_stats = track_df.select(
    "day",
    "recording_id",
    "growth_percentile",
    "cumulative_listen_count",
    "listen_count_past_7_days",
    "listen_pctl_past_7_days",
    "listen_count_past_30_days",
    "listen_pctl_past_30_days"
)

# Write to Postgres
track_stats.write.jdbc(
    url=jdbc_url,
    table="track_daily_stats",
    mode="overwrite",
    properties=jdbc_props
)

print(f"Wrote {track_stats.count():,} track stats records")
track_stats.show(10)

Loaded 7,650,459 track daily records


Wrote 7,650,459 track stats records


+----------+--------------------+-----------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|       day|        recording_id|growth_percentile|cumulative_listen_count|listen_count_past_7_days|listen_pctl_past_7_days|listen_count_past_30_days|listen_pctl_past_30_days|
+----------+--------------------+-----------------+-----------------------+------------------------+-----------------------+-------------------------+------------------------+
|2005-03-23|_Arboria (Planet ...|              0.0|                      2|                       2|                    0.0|                        2|                     0.0|
|2005-03-23|_Ming's Theme (In...|              0.0|                      2|                       2|                    0.0|                        2|                     0.0|
|2005-03-23|           _The Hero|              0.0|                      2|                       2|                    

In [58]:
# Stop Spark session
spark.stop()
print("Spark session stopped.")

Spark session stopped.


## 7) Sanity checks from SQL

In [59]:
import psycopg2

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)

with conn, conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM track_daily_listens;")
    track_ct = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM artist_daily_listens;")
    artist_ct = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM track_daily_stats;")
    track_stats_ct = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM artist_daily_stats;")
    artist_stats_ct = cur.fetchone()[0]
    cur.execute("SELECT * FROM ingestion_state WHERE id=1;")
    state = cur.fetchone()

conn.close()
print({
    "track_daily_listens_rows": track_ct, 
    "artist_daily_listens_rows": artist_ct,
    "track_daily_stats_rows": track_stats_ct,
    "artist_daily_stats_rows": artist_stats_ct
})
print("ingestion_state:", state)

{'track_daily_listens_rows': 7650459, 'artist_daily_listens_rows': 66395, 'track_daily_stats_rows': 7650459, 'artist_daily_stats_rows': 66395}
ingestion_state: (1, '2433', '/Users/didiermunezero/Documents/NU/Junior/DE 300/de300-2026wi-munezero/Final Project/datadumps/musicbrainz/listenbrainz-listens-dump-2433-20260217-000003-incremental.tar.zst', datetime.datetime(2026, 2, 23, 0, 4, 4, 437731, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))))


In [60]:
# Sample stats to verify computation
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host=PGHOST, port=PGPORT, dbname=PGDATABASE, user=PGUSER, password=PGPASSWORD
)

# Top artists by cumulative listens (most recent day)
artist_stats_query = '''
    SELECT a.day, ai.artist_name, a.cumulative_listen_count, 
           a.listen_count_past_7_days, a.growth_percentile,
           a.listen_pctl_past_7_days
    FROM artist_daily_stats a
    JOIN artist_info ai ON a.artist_mbid = ai.artist_mbid
    WHERE a.day = (SELECT MAX(day) FROM artist_daily_stats)
    ORDER BY a.cumulative_listen_count DESC
    LIMIT 10;
'''

# Top tracks by 7-day momentum  
track_stats_query = '''
    SELECT t.day, ti.track_name, t.listen_count_past_7_days,
           t.cumulative_listen_count, t.growth_percentile,
           t.listen_pctl_past_7_days
    FROM track_daily_stats t
    JOIN track_info ti ON t.recording_id = ti.recording_id
    WHERE t.day = (SELECT MAX(day) FROM track_daily_stats)
    ORDER BY t.listen_count_past_7_days DESC
    LIMIT 10;
'''

print("Top Artists (by cumulative listens):")
df_artist_stats = pd.read_sql(artist_stats_query, conn)
display(df_artist_stats)

print("\nTop Tracks (by 7-day listens):")
df_track_stats = pd.read_sql(track_stats_query, conn)
display(df_track_stats)

conn.close()

Top Artists (by cumulative listens):


/var/folders/70/kzs026sn2yj379rclfls772h0000gn/T/ipykernel_8152/749733715.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_artist_stats = pd.read_sql(artist_stats_query, conn)


,day,artist_name,cumulative_listen_count,listen_count_past_7_days,growth_percentile,listen_pctl_past_7_days
0,2026-02-17,Cocteau Twins,173,173,0.500000,1.000000
1,2026-02-17,Kid Cudi,127,127,0.000000,0.833333
2,2026-02-17,Beach House,103,103,0.166667,0.666667
3,2026-02-17,PEEKABOO,36,36,0.333333,0.500000
4,2026-02-17,Marcy Playground,34,34,0.833333,0.333333
5,2026-02-17,Chumbawamba,17,17,0.666667,0.166667
6,2026-02-17,Pegboard Nerds & Grabbitz,11,11,1.000000,0.000000



Top Tracks (by 7-day listens):


/var/folders/70/kzs026sn2yj379rclfls772h0000gn/T/ipykernel_8152/749733715.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_track_stats = pd.read_sql(track_stats_query, conn)


,day,track_name,listen_count_past_7_days,cumulative_listen_count,growth_percentile,listen_pctl_past_7_days
0,2026-02-17,Money,315,2075,0.411765,1.000000
1,2026-02-17,Diet Mountain Dew,80,250,0.470588,0.941176
2,2026-02-17,Lazy Calm,15,35,0.529412,0.882353
3,2026-02-17,Eyes Are Mosaics,12,16,0.588235,0.823529
4,2026-02-17,Tubthumping,9,9,0.647059,0.764706
5,2026-02-17,"Road, River and Rail",7,7,0.705882,0.705882
6,2026-02-17,The Vampires of New York,5,5,0.764706,0.647059
7,2026-02-17,The Shadow of Seattle,2,2,0.823529,0.411765
8,2026-02-17,Lorelei,2,2,0.823529,0.411765
9,2026-02-17,Carolyn’s Fingers,2,2,0.823529,0.411765
